# Riesgo de crédito en un banco
## Notebook 02 · Construcción de variables con SQL

**Entrada:** `data/banco.db`, generada en `01_exploracion.ipynb`.

**Qué hace este notebook:** para cada préstamo, resume el comportamiento de la cuenta antes de la fecha del préstamo: cuántas transacciones hizo, su saldo promedio y mínimo, cuántas veces quedó en negativo y cuántos intereses sancionatorios pagó.

**Regla principal (anti-fuga de información):** solo se usan transacciones con fecha **anterior** a la del préstamo. El banco, al aprobar el préstamo, no conocía lo que pasaría después.

**Resultado:** tabla `prestamos_variables` en la base de datos y `data/processed/prestamos_variables.csv`.

In [1]:
import os
import sqlite3
import pandas as pd

conexion = sqlite3.connect("../data/banco.db")

def consulta(sql):
    """Ejecuta una consulta SQL y devuelve el resultado como DataFrame."""
    return pd.read_sql(sql, conexion)

consulta("SELECT COUNT(*) AS prestamos FROM loan")

,prestamos
0,682


## 1. Historial de cada préstamo

Primero se unen los préstamos con las transacciones de su cuenta, pero solo las anteriores a la fecha del préstamo. La condición de fecha va dentro del `ON` del `JOIN`.

In [2]:
# Primero confirmo el nombre de las columnas de prestamos (loan.csv) y transacciones (trans.csv)
print("Columnas de prestamos: ")
consulta("PRAGMA table_info('loan')")



Columnas de prestamos: 


,cid,name,type,notnull,dflt_value,pk
0,0,loan_id,INTEGER,0,None,0
1,1,account_id,INTEGER,0,None,0
2,2,date,TEXT,0,None,0
3,3,amount,INTEGER,0,None,0
4,4,duration,INTEGER,0,None,0
5,5,payments,REAL,0,None,0
6,6,status,TEXT,0,None,0
7,7,malo,INTEGER,0,None,0


In [3]:
print("Columnas de transacciones: ")
consulta("PRAGMA table_info('trans')")

Columnas de transacciones: 


,cid,name,type,notnull,dflt_value,pk
0,0,trans_id,INTEGER,0,None,0
1,1,account_id,INTEGER,0,None,0
2,2,date,TEXT,0,None,0
3,3,type,TEXT,0,None,0
4,4,operation,TEXT,0,None,0
5,5,amount,INTEGER,0,None,0
6,6,balance,INTEGER,0,None,0
7,7,k_symbol,TEXT,0,None,0
8,8,bank,TEXT,0,None,0
9,9,account,REAL,0,None,0


Identifico por cual fila conectar `account_id` y hago el JOIN LEFT para poder quedarme con la info de transacciones y lo que necesite de prestamos.

In [4]:
consulta("""
SELECT
    l.loan_id,
    l.date              AS fecha_prestamo,
    COUNT(t.trans_id)   AS n_transacciones,
    MIN(t.date)         AS primera_transaccion,
    MAX(t.date)         AS ultima_transaccion
FROM loan AS l
LEFT JOIN trans AS t
    ON t.account_id = l.account_id
    AND t.date < l.date
GROUP BY l.loan_id, l.date
ORDER BY n_transacciones
LIMIT 10
""")

,loan_id,fecha_prestamo,n_transacciones,primera_transaccion,ultima_transaccion
0,5063,1997-07-10,2,1997-03-30,1997-06-28
1,5314,1993-07-05,4,1993-03-22,1993-06-20
2,6594,1994-09-28,5,1994-05-17,1994-09-14
3,5253,1997-06-03,10,1997-02-20,1997-05-31
4,5301,1996-07-06,10,1996-03-14,1996-06-30
5,5892,1996-05-14,10,1995-12-27,1996-04-30
6,6736,1994-06-14,10,1994-02-27,1994-05-31
7,5765,1998-02-10,11,1997-10-18,1998-02-08
8,5973,1997-11-24,11,1997-08-08,1997-11-10
9,6354,1996-10-23,11,1996-06-25,1996-10-12


---
## 2. Variables de comportamiento

Para cada préstamo se calculan, con su historial previo:

| Variable | Qué mide |
|---|---|
| `n_transacciones` | Actividad de la cuenta |
| `meses_historial` | Antigüedad de la cuenta al pedir el préstamo |
| `saldo_promedio`, `saldo_minimo` | Capacidad financiera |
| `n_saldo_negativo` | Veces que la cuenta quedó en negativo |
| `n_interes_sancion` | Intereses sancionatorios cobrados |
| `ingreso_promedio`, `egreso_promedio` | Monto típico de ingresos y egresos |
| `saldo_final` | Saldo justo antes del préstamo |

`saldo_final` usa una **función de ventana**: `ROW_NUMBER()` numera las transacciones de cada préstamo de la más reciente a la más antigua, y se toma la número 1.

In [5]:
variables = consulta("""
WITH historial AS (
    SELECT
        l.loan_id,
        t.date,
        t.amount,
        t.balance,
        t.type,
        t.k_symbol,
        ROW_NUMBER() OVER (
            PARTITION BY l.loan_id
            ORDER BY t.date DESC, t.trans_id DESC
        ) AS orden_reciente
    FROM loan AS l
    JOIN trans AS t
        ON  t.account_id = l.account_id
        AND t.date < l.date
),
resumen AS (
    SELECT
        loan_id,
        COUNT(*)                                                     AS n_transacciones,
        ROUND(AVG(balance), 0)                                       AS saldo_promedio,
        MIN(balance)                                                 AS saldo_minimo,
        SUM(CASE WHEN balance < 0 THEN 1 ELSE 0 END)                 AS n_saldo_negativo,
        SUM(CASE WHEN k_symbol = 'Interés sancionatorio' THEN 1 ELSE 0 END) AS n_interes_sancion,
        ROUND(AVG(CASE WHEN type = 'Ingreso' THEN amount END), 0)    AS ingreso_promedio,
        ROUND(AVG(CASE WHEN type <> 'Ingreso' THEN amount END), 0)   AS egreso_promedio,
        MAX(CASE WHEN orden_reciente = 1 THEN balance END)           AS saldo_final
    FROM historial
    GROUP BY loan_id
)
SELECT
    l.loan_id,
    l.account_id,
    l.date AS fecha_prestamo,
    l.amount   AS monto,
    l.duration AS plazo_meses,
    l.payments AS cuota,
    ROUND((julianday(l.date) - julianday(a.date)) / 30.44, 1) AS meses_historial,
    COALESCE(r.n_transacciones, 0) AS n_transacciones,
    r.saldo_promedio,
    r.saldo_minimo,
    COALESCE(r.n_saldo_negativo, 0)  AS n_saldo_negativo,
    COALESCE(r.n_interes_sancion, 0) AS n_interes_sancion,
    r.ingreso_promedio,
    r.egreso_promedio,
    r.saldo_final,
    l.malo
FROM loan AS l
JOIN account AS a
    ON a.account_id = l.account_id
LEFT JOIN resumen AS r
    ON r.loan_id = l.loan_id
""")

assert len(variables) == 682, "Se perdieron o duplicaron préstamos"
variables.head()

,loan_id,account_id,fecha_prestamo,monto,plazo_meses,cuota,meses_historial,n_transacciones,saldo_promedio,saldo_minimo,n_saldo_negativo,n_interes_sancion,ingreso_promedio,egreso_promedio,saldo_final,malo
0,4959,2,1994-01-05,80952,24,3373.0,10.3,54,32591.0,1100,0,0,10395.0,6276.0,27855,0
1,4961,19,1996-04-29,30276,12,2523.0,12.7,80,25197.0,715,0,0,5265.0,6657.0,15854,1
2,4962,25,1997-12-08,30276,12,2523.0,16.4,164,62991.0,900,0,0,19787.0,8651.0,31622,0
3,4967,37,1998-10-14,318480,60,5308.0,13.9,116,39954.0,-1011,1,1,13980.0,5131.0,16368,1
4,4968,38,1998-04-19,110736,48,2307.0,8.3,55,31384.0,13841,0,0,6452.0,3468.0,37417,0
